In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Excel dosyasını okuma
file_path = '/mnt/data/Project_Data.xlsx'
data = pd.read_excel(file_path)

# Saatlik sütunları seçme (ilk sütun hariç)
hourly_columns = data.columns[1:]

# İlk sütunu cihaz isimleri olarak ayarlama
data_excluding_total = data.iloc[:-1].copy()
data_excluding_total.set_index(data.columns[0], inplace=True)

# Tüm cihazların saatlik enerji tüketimini toplama
daily_load_profile_corrected = data_excluding_total[hourly_columns].sum()

# Eşik değer tanımlama
threshold = 7500  # Eşik değer (Watt cinsinden)

# Toplam günlük yük (kWh)
total_daily_load_kwh = daily_load_profile_corrected.sum() / 1000  # Watt'tan kWh'ye dönüştürme

# Saatlik maliyet hesaplama için tarifeler
tariffs = {
    'T1': 3.15,  # Gündüz: 06:00 - 17:00
    'T2': 4.62,  # Yoğun: 17:00 - 22:00
    'T3': 1.97   # Gece: 22:00 - 06:00
}

time_periods = {
    'T1': range(6, 17),  # Gündüz saatleri
    'T2': range(17, 22),  # Yoğun saatler
    'T3': list(range(22, 24)) + list(range(0, 6))  # Gece saatleri (gece yarısını içerir)
}

hourly_cost = []
for i, hour in enumerate(hourly_columns):
    hour_int = int(hour.split('-')[0].split(':')[0])  # Saat bilgisini ayıkla
    for period, hours in time_periods.items():
        if hour_int in hours:
            rate = tariffs[period]  # Bu saat için tarifeyi bul
            break
    energy_kwh = daily_load_profile_corrected[hour] / 1000  # Watt'tan kWh'ye dönüştürme
    cost = energy_kwh * rate
    hourly_cost.append(cost)

# Toplam günlük maliyet
total_daily_cost = sum(hourly_cost)

# Spesifik yük kaydırma işlemi
shifted_profile = daily_load_profile_corrected.copy()

# 17:00-19:00 arasında Music System yükü 22:00-24:00 aralığına taşınıyor
shifted_profile['17:00-18:00'] -= data_excluding_total.at['Music System (Low)', '17:00-18:00']
shifted_profile['22:00-23:00'] += data_excluding_total.at['Music System (Low)', '17:00-18:00']

shifted_profile['18:00-19:00'] -= data_excluding_total.at['Music System (Low)', '18:00-19:00']
shifted_profile['23:00-24:00'] += data_excluding_total.at['Music System (Low)', '18:00-19:00']

# 18:00-20:00 arasında Computer and Monitor yükü 22:00-24:00 aralığına taşınıyor
shifted_profile['18:00-19:00'] -= data_excluding_total.at['Computer and Monitor (Low)', '18:00-19:00']
shifted_profile['22:00-23:00'] += data_excluding_total.at['Computer and Monitor (Low)', '18:00-19:00']

shifted_profile['19:00-20:00'] -= data_excluding_total.at['Computer and Monitor (Low)', '19:00-20:00']
shifted_profile['23:00-24:00'] += data_excluding_total.at['Computer and Monitor (Low)', '19:00-20:00']

# 19:00-21:00 arasında Dishwasher yükü 22:00-24:00 aralığına taşınıyor
shifted_profile['19:00-20:00'] -= data_excluding_total.at['Dishwasher (Low)', '19:00-20:00']
shifted_profile['22:00-23:00'] += data_excluding_total.at['Dishwasher (Low)', '19:00-20:00']

shifted_profile['20:00-21:00'] -= data_excluding_total.at['Dishwasher (Low)', '20:00-21:00']
shifted_profile['23:00-24:00'] += data_excluding_total.at['Dishwasher (Low)', '20:00-21:00']

# Kaydırılmış yük profili için toplam maliyet hesaplama
shifted_hourly_cost = []
for i, hour in enumerate(hourly_columns):
    hour_int = int(hour.split('-')[0].split(':')[0])  # Saat bilgisini ayıkla
    for period, hours in time_periods.items():
        if hour_int in hours:
            rate = tariffs[period]  # Bu saat için tarifeyi bul
            break
    energy_kwh = shifted_profile[hour] / 1000  # Watt'tan kWh'ye dönüştürme
    cost = energy_kwh * rate
    shifted_hourly_cost.append(cost)

# Kaydırılmış profil için toplam maliyet
shifted_total_daily_cost = sum(shifted_hourly_cost)

# 1. Orijinal Yük Profili Grafiği
plt.figure(figsize=(12, 6))
plt.step(hourly_columns, daily_load_profile_corrected, where='mid', label='Original Energy Use (W)', color='orange')
plt.axhline(y=threshold, color='red', linestyle='--', label='Threshold Value (7500 W)')
plt.text(0.02, 0.9, f'Total Load: {total_daily_load_kwh:.2f} kWh',
         transform=plt.gca().transAxes, fontsize=12, color='green', bbox=dict(facecolor='white', alpha=0.8))
plt.text(0.02, 0.8, f'Total Cost: {total_daily_cost:.2f} TL',
         transform=plt.gca().transAxes, fontsize=12, color='blue', bbox=dict(facecolor='white', alpha=0.8))
plt.ylim(0, 11000)
plt.yticks(range(0, 11001, 1000))
plt.title('Original Load Profile')
plt.xlabel('Hours')
plt.ylabel('Energy Use (W)')
plt.xticks(ticks=range(len(hourly_columns)), labels=hourly_columns, rotation=45, ha='right')  # Saat etiketleri 45 derece
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

# 2. Kaydırılmış Yük Profili Grafiği
plt.figure(figsize=(12, 6))
plt.step(hourly_columns, shifted_profile, where='mid', label='Shifted Energy Usage (W)', color='blue')
plt.axhline(y=threshold, color='red', linestyle='--', label='Threshold Value (7500 W)')
plt.text(0.02, 0.9, f'Total Load: {total_daily_load_kwh:.2f} kWh',
         transform=plt.gca().transAxes, fontsize=12, color='green', bbox=dict(facecolor='white', alpha=0.8))
plt.text(0.02, 0.8, f'Total Cost: {shifted_total_daily_cost:.2f} TL',
         transform=plt.gca().transAxes, fontsize=12, color='blue', bbox=dict(facecolor='white', alpha=0.8))
plt.ylim(0, 10000)
plt.yticks(range(0, 10001, 1000))
plt.title('Shifted Load Profile')
plt.xlabel('Hours')
plt.ylabel('Energy Use (W)')
plt.xticks(ticks=range(len(hourly_columns)), labels=hourly_columns, rotation=45, ha='right')  # Saat etiketleri 45 derece
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

# 3. Orijinal ve Kaydırılmış Profilleri Karşılaştırma Grafiği
plt.figure(figsize=(12, 6))
plt.title('Comparison of the Original and Shifted Load Profile')
plt.step(hourly_columns, daily_load_profile_corrected, where='mid', label='Original Energy Use (W)', color='orange')
plt.step(hourly_columns, shifted_profile, where='mid', label='Shifted Energy Usage (W)', color='blue')
plt.axhline(y=threshold, color='red', linestyle='--', label='Threshold Value (7500 W)')
plt.text(0.02, 0.9, f'Total Cost: {total_daily_load_kwh:.2f} kWh',
         transform=plt.gca().transAxes, fontsize=12, color='green', bbox=dict(facecolor='white', alpha=0.8))
plt.text(0.02, 0.8, f'Saving: {total_daily_cost - shifted_total_daily_cost:.2f} TL',
         transform=plt.gca().transAxes, fontsize=12, color='purple', bbox=dict(facecolor='white', alpha=0.8))

FileNotFoundError: [Errno 2] No such file or directory: '/mnt/data/Project_Data.xlsx'